# SafeQuest fine-tune (Gemma 3 + LoRA, Colab free T4)

Spec §7. Input: `train.jsonl` and `val.jsonl` from `npm run data -- build`.
Output: a LoRA adapter, a GGUF for Ollama, the training log and the loss curve (backup slide).

**Runtime → Change runtime type → T4 GPU**, then run top to bottom.
Before uploading, run `npm run eval -- check-leak data/out/train.jsonl` locally.

In [ ]:
%%capture
!pip install unsloth

## 1. Settings

Pick the model size from the day-1 latency test on the demo laptop (spec §7): use 4B if it generates a scene in under 8 s through Ollama, otherwise 1B.

In [ ]:
MODEL = "unsloth/gemma-3-4b-it"   # or "unsloth/gemma-3-1b-it"
MAX_SEQ_LEN = 2048
EPOCHS = 2
LORA_R = 16
GGUF_QUANT = "Q8_0"               # fast to export; "q4_k_m" is smaller/faster on the laptop but builds llama.cpp first
OUT = "safequest_ft"

## 2. Upload the dataset

Use the file picker (or mount Drive and point to the files).

In [ ]:
import json, os
if not os.path.exists("train.jsonl"):
    from google.colab import files
    files.upload()   # select train.jsonl and val.jsonl

def read_jsonl(path):
    if not os.path.exists(path):
        return []
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

train_rows = read_jsonl("train.jsonl")
val_rows = read_jsonl("val.jsonl")
print(len(train_rows), "train /", len(val_rows), "val examples")
assert train_rows, "train.jsonl is empty"

## 3. Load Gemma 3 with LoRA

In [ ]:
from unsloth import FastModel
import torch

model, tokenizer = FastModel.from_pretrained(
    model_name = MODEL,
    max_seq_length = MAX_SEQ_LEN,
    load_in_4bit = True,
    full_finetuning = False,
)
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers = False,
    finetune_language_layers = True,
    finetune_attention_modules = True,
    finetune_mlp_modules = True,
    r = LORA_R,
    lora_alpha = LORA_R,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

## 4. Render the prompt format

This must match **exactly** what Ollama sends at demo time (`training/Modelfile`):
the system prompt is folded into the first user turn, as Gemma 3 has no system role.
The assert checks it against the tokenizer's own Gemma 3 chat template.

In [ ]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(tokenizer, chat_template = "gemma-3")

def render_prompt(system, user):
    return f"<start_of_turn>user\n{system}\n\n{user}<end_of_turn>\n<start_of_turn>model\n"

def render(row):
    m = {x["role"]: x["content"] for x in row["messages"]}
    return render_prompt(m["system"], m["user"]) + m["assistant"] + "<end_of_turn>\n"

# Same text as the official template (minus <bos>, which the tokenizer adds itself).
reference = tokenizer.apply_chat_template(train_rows[0]["messages"], tokenize = False).removeprefix("<bos>")
assert render(train_rows[0]) == reference, (render(train_rows[0]), reference)
print(render(train_rows[0])[:400])

In [ ]:
from datasets import Dataset
train_ds = Dataset.from_list([{"text": render(r)} for r in train_rows])
val_ds = Dataset.from_list([{"text": render(r)} for r in val_rows]) if val_rows else None
lengths = [len(tokenizer(text = t)["input_ids"]) for t in train_ds["text"][:200]]
print("max tokens in first 200:", max(lengths), "(MAX_SEQ_LEN =", MAX_SEQ_LEN, ")")

## 5. Train (loss only on the model's answer)

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_ds,
    eval_dataset = val_ds,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        num_train_epochs = EPOCHS,
        learning_rate = 2e-4,
        logging_steps = 5,
        eval_strategy = "steps" if val_ds else "no",
        eval_steps = 25,
        per_device_eval_batch_size = 2,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none",
        output_dir = "checkpoints",
    ),
)
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<start_of_turn>user\n",
    response_part = "<start_of_turn>model\n",
)
stats = trainer.train()
print(stats)

## 6. Save the training log and loss curve (backup slide)

In [ ]:
import csv
import matplotlib.pyplot as plt

log = trainer.state.log_history
with open("training_log.csv", "w", newline = "") as f:
    w = csv.writer(f)
    w.writerow(["step", "train_loss", "eval_loss"])
    for e in log:
        if "loss" in e or "eval_loss" in e:
            w.writerow([e.get("step"), e.get("loss", ""), e.get("eval_loss", "")])

train_pts = [(e["step"], e["loss"]) for e in log if "loss" in e]
eval_pts = [(e["step"], e["eval_loss"]) for e in log if "eval_loss" in e]
plt.figure(figsize = (8, 4.5))
plt.plot(*zip(*train_pts), label = "train loss")
if eval_pts:
    plt.plot(*zip(*eval_pts), "o-", label = "val loss")
plt.xlabel("step"); plt.ylabel("loss"); plt.title(f"SafeQuest fine-tune · {MODEL.split('/')[-1]} · {len(train_rows)} examples")
plt.legend(); plt.grid(alpha = 0.3); plt.tight_layout()
plt.savefig("loss_curve.png", dpi = 160)
plt.show()

## 7. Sanity check on val prompts

Only checks that output is JSON. The real scoring is `npm run eval` after importing into Ollama.

In [ ]:
FastModel.for_inference(model) if hasattr(FastModel, "for_inference") else None
ok = 0
sample = (val_rows or train_rows)[:10]
for row in sample:
    m = {x["role"]: x["content"] for x in row["messages"]}
    inputs = tokenizer(text = [render_prompt(m["system"], m["user"])], return_tensors = "pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens = 900, do_sample = False)
    reply = tokenizer.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens = True)[0].strip()
    try:
        json.loads(reply); ok += 1; mark = "ok "
    except Exception:
        mark = "BAD"
    print(mark, m["user"].replace("\n", " | ")[:70], "->", reply[:90])
print(f"{ok}/{len(sample)} parse as JSON")

## 8. Export

* `safequest_ft_lora/`: the adapter (small; keep it).
* GGUF for Ollama on the demo laptop.
* Optional: push merged weights to the Hugging Face Hub for the hosted endpoint.

In [ ]:
model.save_pretrained(f"{OUT}_lora")
tokenizer.save_pretrained(f"{OUT}_lora")
model.save_pretrained_gguf(OUT, tokenizer, quantization_method = GGUF_QUANT)
import glob
print(glob.glob(f"{OUT}*/*.gguf") + glob.glob("*.gguf"))

In [ ]:
# Optional: hosted endpoint (spec §7). Needs a HF write token in Colab secrets as HF_TOKEN.
if False:
    from google.colab import userdata
    model.push_to_hub_merged("YOUR_HF_USER/safequest-gemma3-ft", tokenizer, token = userdata.get("HF_TOKEN"))

In [ ]:
# Download: GGUF can be several GB. Copying to Drive is usually faster than the browser download.
if False:
    from google.colab import drive
    drive.mount("/content/drive")
    !mkdir -p /content/drive/MyDrive/safequest && cp -r {OUT}*/*.gguf {OUT}_lora training_log.csv loss_curve.png /content/drive/MyDrive/safequest/
else:
    from google.colab import files
    files.download("loss_curve.png"); files.download("training_log.csv")